# 09 · KG + LLM integration — realistic drug-repurposing ranking

**Task.** For a target disease, an LLM ranks a pool of candidate drugs by
repurposing plausibility. The true (post-cutoff) repurposed drug is hidden among
distractors; we score the **rank** it receives (MRR, hits@k). The only thing that
changes between arms is the knowledge-graph evidence attached to each candidate,
so any lift over the no-KG baseline is attributable to the KG.

This replaces the earlier binary yes/no plausibility design. What is new:

- **Listwise ranking**, not yes/no — closer to how a pharma analyst triages.
- **Positives from the prospective (time-split) gold standard**, so the answer is
  not already in the model's parametric memory.
- **Reason-then-JSON prompt**: free-text reasoning precedes a strict JSON block;
  the KG is attributed as *a source*, not ground truth; abstention is allowed;
  each candidate carries `basis` / `evidence_agreement` / `confidence` fields so
  we can measure whether the model *used* the KG and whether it over-trusted it.
- **Balanced dual dossiers**: each drug gets a neutral, query-independent profile
  (targets, pathways, indications, side-effects) and the disease gets one profile
  (associated genes, phenotypes). The model must connect them itself — no
  pre-computed drug→disease bridge (that would leak the answer).
- **Position-bias control**: candidate order is reshuffled across runs and the
  rank is aggregated over shuffles.
- **Model slate**: runs the same task across local Ollama models *and* hosted
  APIs (Anthropic / OpenAI / Groq / OpenRouter / Gemini).

**Modes.** `MODE = 'mock'` exercises the whole pipeline (pools → prompt → shuffles
→ parse → score) with a mock model and no KG load, so it runs anywhere. `MODE =
'real'` calls the configured models and builds KG dossiers from the real graphs.


## 1 · Setup

In [ ]:
import os, sys, json, re, string, time, gc
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Repo root = nearest parent containing config/kgs.yaml (portable across machines).
BASE = Path.cwd().resolve()
while not (BASE / 'config' / 'kgs.yaml').exists() and BASE != BASE.parent:
    BASE = BASE.parent
sys.path.insert(0, str(BASE / 'src'))

# Load secrets from biokgsuite/.env (gitignored) into the environment, if present.
_envf = BASE / '.env'
if _envf.exists():
    for _line in _envf.read_text().splitlines():
        _line = _line.strip()
        if _line and not _line.startswith('#') and '=' in _line:
            _k, _v = _line.split('=', 1)
            os.environ.setdefault(_k.strip(), _v.strip().strip('\"').strip("'"))

from loading import load_kg, load_config, find_config        # noqa: E402
from graph_utils import build_lookup_maps                    # noqa: E402
from plotting import setup_style, save_fig                   # noqa: E402

config = load_config(find_config(BASE))
GOLD   = BASE / 'data' / 'gold_standards'
FIGS   = BASE / 'results' / 'figures';                FIGS.mkdir(parents=True, exist_ok=True)
TABLES = BASE / 'results' / 'tables' / '09_llm_runs'; TABLES.mkdir(parents=True, exist_ok=True)

setup_style()
LETTERS = list(string.ascii_uppercase)
print(f'repo root: {BASE}')

## 2 · Configuration

Everything you would normally pass on the command line lives here. Start in
`'mock'` to check the plumbing, then switch to `'real'`. Hosted-API models read
their key from the matching environment variable (`ANTHROPIC_API_KEY`,
`OPENAI_API_KEY`, `GROQ_API_KEY`, `OPENROUTER_API_KEY`, `GEMINI_API_KEY`); local
models must be pulled in Ollama (`ollama pull <name>`).

In [ ]:
# ── Run mode ────────────────────────────────────────────────────────────────
MODE = 'real'
MOCK = (MODE == 'mock')
LOAD_COMPILED = True   # run 09 from the compiled per-LLM results (no model calls)
FORCE_RERUN = False

# ── Model slate (hosted APIs; run ONE per launch via BKG_MODELS) ────────────
MODELS = [
    'gpt-4.1-mini',                  # mid rung        · OPENAI_API_KEY (paid, cheap)
    'gemini:gemini-3.5-flash-lite',  # mid closed (Google lineage) - GEMINI_API_KEY, cheap
    'groq:llama-3.3-70b-versatile',  # frontier open   · GROQ_API_KEY  (free)
]

# ── KGs + experimental arms ─────────────────────────────────────────────────
KGS        = ['primekg', 'drkg', 'biokg']   # openbilink dropped; fixed set evaluable in all 3
CONDITIONS = ['no_kg', 'kg']
BRIDGE_MODE = 'full'
CAP         = 8
ANONYMIZE       = True
ANONYMIZE_GENES = False
if ANONYMIZE_GENES:
    ANONYMIZE = True

# ── Sampling + decoding ─────────────────────────────────────────────────────
GOLD_FILE   = GOLD / 'legacy' / 'gold_standard_v3_common3.tsv'   # 76 pairs evaluable in ALL 3 KGs (fixed set)
N_DISEASES  = 1000     # cap; per-pair uses all rows (76)
POOL_SIZE   = 8
SHUFFLES    = 3
STRATIFY    = True
UNIQUE_DRUG = True
PER_PAIR    = True       # one query per drug-disease PAIR -> uses all 76 rows (diseases may repeat)
DISTRACTOR_POOL = 'opentargets_approved'   # fixed pool; exclude drugs indicated for the disease
TEMPERATURE = 0.7
API_DELAY   = 1.0
SEED        = 0
SEEDS       = [0, 1, 2]   # 3 seeds x 3 shuffles = 9 evals per (disease, KG, condition) cell

OUT_CSV = TABLES / '09_ranking.csv'

# ── Optional per-run overrides via env (no further edits needed) ─────────────
import os as _os
if _os.environ.get('BKG_MODELS'):
    MODELS = [x.strip() for x in _os.environ['BKG_MODELS'].split(',') if x.strip()]
if _os.environ.get('BKG_OUT_CSV'):
    OUT_CSV = TABLES / _os.environ['BKG_OUT_CSV']
if _os.environ.get('BKG_SEEDS'):
    SEEDS = [int(x) for x in _os.environ['BKG_SEEDS'].split(',') if x.strip() != '']
if _os.environ.get('BKG_SHUFFLES'):
    SHUFFLES = int(_os.environ['BKG_SHUFFLES'])
if _os.environ.get('BKG_NDISEASES'):
    N_DISEASES = int(_os.environ['BKG_NDISEASES'])
if _os.environ.get('BKG_GOLD'):
    GOLD_FILE = GOLD / 'legacy' / _os.environ['BKG_GOLD']

print(f'MODE={MODE}  models={MODELS}')
print(f'GOLD={GOLD_FILE.name}  KGS={KGS}  n_diseases={N_DISEASES}  seeds={SEEDS}  shuffles={SHUFFLES}')



## 3 · Prompt (reason-then-JSON) and output schema

The prompt asks the model to weigh KG evidence against its own pharmacological
knowledge, flag conflicts, and abstain when evidence is insufficient — then emit
a single JSON object. The schema makes every reliance field required, so we can
measure how much the model leaned on the KG rather than only the final ranking.

In [ ]:
PROMPT_HEADER = (
    "You are assisting a drug repurposing scientist. For the target disease below,\n"
    "you are given candidate drugs. Some include evidence retrieved from a\n"
    "biomedical knowledge graph, reported as mechanistic links\n"
    "(drug -> target -> pathway -> disease) and related annotations.\n"
)
PROMPT_INSTRUCTIONS = (
    "Assess each candidate's mechanistic plausibility for this disease using BOTH\n"
    "the knowledge-graph evidence and your own pharmacological knowledge. The\n"
    "knowledge graph reports associations; treat them as one source, not as\n"
    "established truth. Where its evidence is sparse, or conflicts with what you\n"
    "know, say so and weight it accordingly -- do not construct a rationale to fit\n"
    "weak evidence. If the evidence is insufficient to judge a candidate, mark it\n"
    "\"uncertain\" rather than forcing a confident rank.\n\n"
    "Respond with ONLY a single JSON object (no text before or after) of the form:\n"
    "{ \"reasoning\": \"<a few sentences weighing the candidates and flagging any "
    "KG-vs-knowledge conflicts>\",\n"
    "  \"ranking\": [ { \"rank\": 1, \"drug\": \"<letter>\",\n"
    "    \"basis\": \"kg\" | \"prior\" | \"both\",\n"
    "    \"evidence_agreement\": \"consistent\" | \"conflicting\" | \"insufficient\",\n"
    "    \"confidence\": 1-5, \"rationale\": \"<one sentence>\" }, ... ] }\n"
    "Rank every candidate. ALL fields are required for EVERY candidate, even when "
    "no evidence is shown: set basis=\"prior\" when you used only your own "
    "knowledge (\"kg\"/\"both\" when you used the provided evidence), "
    "evidence_agreement=\"insufficient\" when no evidence was given; confidence "
    "(1-5) and a one-sentence rationale are ALWAYS required."
)

# JSON schema for enforced structured output (Ollama `format=`, OpenAI json_schema).
RANKING_SCHEMA = {
    "type": "object",
    "properties": {
        "reasoning": {"type": "string"},
        "ranking": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "rank": {"type": "integer"},
                    "drug": {"type": "string"},
                    "basis": {"type": "string", "enum": ["kg", "prior", "both"]},
                    "evidence_agreement": {"type": "string",
                                           "enum": ["consistent", "conflicting", "insufficient"]},
                    "confidence": {"type": "integer", "minimum": 1, "maximum": 5},
                    "rationale": {"type": "string"},
                },
                "required": ["rank", "drug", "basis", "evidence_agreement",
                             "confidence", "rationale"],
            },
        },
    },
    "required": ["reasoning", "ranking"],
}


def build_prompt(disease_name, disease_id, candidates, disease_profile=None):
    """candidates: list of dicts {letter, drug_name, evidence (str or None)}.
    disease_profile: KG disease dossier shown once in the header (KG arms only)."""
    lines = [PROMPT_HEADER, f"\nTarget disease: {disease_name} ({disease_id})"]
    if disease_profile:
        lines.append(f"Disease profile (from the knowledge graph): {disease_profile}")
    lines.append("\nCandidates (listed order is arbitrary):")
    for c in candidates:
        lines.append(f"{c['letter']}. {c['drug_name']}")
        if c['evidence']:                       # omit the line entirely if absent
            lines.append(f"   Drug profile (from the knowledge graph): {c['evidence']}")
    lines.append("")
    lines.append(PROMPT_INSTRUCTIONS)
    return "\n".join(lines)

## 4 · Response parsing

`parse_response` recovers the ranked letter order plus the per-candidate reliance
fields. If the JSON is malformed it falls back to first-mention order, and a
missing positive is scored as a miss (`rank = pool_size + 1`).

In [ ]:
def parse_ranking(text, valid_letters):
    """Ordered list of candidate letters, best first. [] if unparseable."""
    if not text:
        return []
    m = re.search(r'\{.*\}', text, re.DOTALL)
    if m:
        try:
            obj = json.loads(m.group(0))
            ranked = []
            for it in obj.get('ranking', []):
                d = str(it.get('drug', '')).strip().upper()
                d = d[0] if d and d[0] in valid_letters else None
                rk = it.get('rank')
                if d and d not in [r[0] for r in ranked]:
                    ranked.append((d, rk if isinstance(rk, (int, float)) else len(ranked) + 1))
            if ranked:
                ranked.sort(key=lambda t: t[1])
                return [d for d, _ in ranked]
        except (json.JSONDecodeError, TypeError, AttributeError):
            pass
    seen = []
    for tok in re.findall(r'\b([A-Z])\b', text.upper()):
        if tok in valid_letters and tok not in seen:
            seen.append(tok)
    return seen


def _as_int(x):
    try:
        return max(1, min(5, int(x)))
    except (ValueError, TypeError):
        return None


def parse_response(text, valid_letters):
    """Like parse_ranking, plus per-letter reliance fields
    {letter: {basis, evidence_agreement, confidence, rationale}}."""
    fields, ordered = {}, []
    if text:
        m = re.search(r'\{.*\}', text, re.DOTALL)
        if m:
            try:
                obj = json.loads(m.group(0))
                ranked = []
                for it in obj.get('ranking', []):
                    d = str(it.get('drug', '')).strip().upper()
                    d = d[0] if d and d[0] in valid_letters else None
                    if not d:
                        continue
                    if d not in fields:
                        fields[d] = {
                            'basis': (str(it.get('basis', '')).lower() or None),
                            'evidence_agreement': (str(it.get('evidence_agreement', '')).lower() or None),
                            'confidence': _as_int(it.get('confidence')),
                            'rationale': (str(it.get('rationale', '')).strip() or None),
                        }
                    if d not in [r[0] for r in ranked]:
                        rk = it.get('rank')
                        ranked.append((d, rk if isinstance(rk, (int, float)) else len(ranked) + 1))
                ranked.sort(key=lambda t: t[1])
                ordered = [d for d, _ in ranked]
            except (json.JSONDecodeError, TypeError, AttributeError):
                pass
    if not ordered:
        ordered = parse_ranking(text, valid_letters)
    return ordered, fields


def rank_of(positive_letter, ordered_letters, pool_size):
    """1-based rank of the positive; pool_size+1 (miss) if absent/unparsed."""
    if positive_letter in ordered_letters:
        return ordered_letters.index(positive_letter) + 1
    return pool_size + 1

## 5 · LLM clients — local Ollama, hosted APIs, and a mock

`rank_call` dispatches a single prompt to the right backend. Hosted models are
addressed as `provider:model_id` (or a bare `claude-*` / `gpt-*`), retry on HTTP
429 with bounded backoff, and cache whichever JSON-format flag the provider
accepts. The mock backend simulates a discerning model so the parser and scorer
are exercised without any network call.

In [ ]:
OLLAMA_URL = 'http://localhost:11434/api/generate'
TAGS_URL   = 'http://localhost:11434/api/tags'

API_PROVIDERS = {
    'anthropic':  dict(url='https://api.anthropic.com/v1/messages',
                       key='ANTHROPIC_API_KEY', style='anthropic'),
    'openai':     dict(url='https://api.openai.com/v1/chat/completions',
                       key='OPENAI_API_KEY', style='openai'),
    'groq':       dict(url='https://api.groq.com/openai/v1/chat/completions',
                       key='GROQ_API_KEY', style='openai'),
    'openrouter': dict(url='https://openrouter.ai/api/v1/chat/completions',
                       key='OPENROUTER_API_KEY', style='openai'),
    'gemini':     dict(url='https://generativelanguage.googleapis.com/v1beta/openai/chat/completions',
                       key='GEMINI_API_KEY', style='openai'),
    'deepseek':   dict(url='https://api.deepseek.com/v1/chat/completions',
                       key='DEEPSEEK_API_KEY', style='openai'),
}
_REASONING_HINTS = ('gpt-oss', 'o1', 'o3', 'o4', '-r1', 'deepseek-r', 'qwq', 'reason', 'think')
_API_FORMAT_CACHE = {}


def _provider_model(name):
    prefix = name.split(':', 1)[0]
    if prefix in API_PROVIDERS and ':' in name:
        return prefix, name.split(':', 1)[1]
    low = name.lower()
    if low.startswith('claude'):
        return 'anthropic', name
    if low.startswith(('gpt-', 'o1', 'o3', 'o4')):
        return 'openai', name
    return None, None


def is_api_model(name):
    return _provider_model(name)[0] is not None


def _max_tokens(name):
    n = name.lower()
    if any(h in n for h in _REASONING_HINTS): return 8000
    if 'gemini' in n: return 8000   # Gemini 2.5/3.x Flash 'think' by default — budget for thinking + JSON
    return 1500


def _api_key_for(name):
    prov, _ = _provider_model(name)
    return os.environ.get(API_PROVIDERS[prov]['key']) if prov else None


def ollama_models_available():
    import requests
    try:
        data = requests.get(TAGS_URL, timeout=3).json()
        return {m['name'] for m in data.get('models', [])}
    except Exception as e:
        raise SystemExit(f"Ollama not reachable at {TAGS_URL}: {e}")


_OLLAMA_DBG = {'n': 0}
def ollama_rank(model, prompt, seed, temperature):
    import requests
    payload = {'model': model, 'prompt': prompt, 'stream': False,
               'format': 'json',                       # simple JSON mode (robust) instead of strict schema
               'options': {'temperature': temperature,
                           'num_predict': max(_max_tokens(model), 4000),   # 70B is verbose; avoid truncation
                           'num_ctx': 8192, 'seed': int(seed)}}
    try:
        r = requests.post(OLLAMA_URL, json=payload, timeout=600)
        r.raise_for_status()
        resp = r.json().get('response', '')
        if _OLLAMA_DBG['n'] < 2:
            _OLLAMA_DBG['n'] += 1
            print('  [ollama raw sample]', repr(resp[:300]), flush=True)
        return resp
    except Exception as e:
        return f'__ERROR__ {e}'


def _post_with_backoff(url, headers, body, timeout=180, max_retries=8):
    import requests, random
    delay = 3.0
    for _ in range(max_retries):
        try:
            r = requests.post(url, headers=headers, json=body, timeout=timeout)
        except Exception as e:
            time.sleep(min(delay, 30)); delay = min(delay * 2, 60); continue   # retry transient network errors
        if r.status_code == 429:
            ra = r.headers.get('retry-after', '')
            try:
                wait = float(ra)
            except ValueError:
                wait = delay
            time.sleep(min(wait, 60) + random.uniform(0, 1.5))   # honor Retry-After up to 60s, + jitter
            delay = min(delay * 2, 60)
            continue
        return r, None
    return None, '__ERROR__ 429 rate-limited (gave up after retries)'


def api_rank(name, prompt, temperature):
    prov, model = _provider_model(name)
    cfg = API_PROVIDERS.get(prov)
    if not cfg:
        return '__ERROR__ not an API model'
    key = os.environ.get(cfg['key'])
    if not key:
        return f'__ERROR__ no key ({cfg["key"]})'
    mx = _max_tokens(name)
    if cfg['style'] == 'anthropic':
        r, err = _post_with_backoff(cfg['url'],
            {'x-api-key': key, 'anthropic-version': '2023-06-01', 'content-type': 'application/json'},
            {'model': model, 'max_tokens': mx, 'temperature': temperature,
             'messages': [{'role': 'user', 'content': prompt}]})
        if err:
            return err
        if r.status_code >= 400:
            return f'__ERROR__ {r.status_code} {r.text[:160]}'
        try:
            return ''.join(b.get('text', '') for b in r.json().get('content', []))
        except Exception as e:
            return f'__ERROR__ parse {e}'
    headers = {'Authorization': f'Bearer {key}', 'content-type': 'application/json'}
    base = {'model': model, 'max_tokens': mx, 'temperature': temperature,
            'messages': [{'role': 'user', 'content': prompt}]}
    ck = (prov, model)
    formats = ([_API_FORMAT_CACHE[ck]] if ck in _API_FORMAT_CACHE
               else [{'type': 'json_object'}, None])
    last = '__ERROR__ no response'
    for fmt in formats:
        body = {**base, **({'response_format': fmt} if fmt else {})}
        r, err = _post_with_backoff(cfg['url'], headers, body)
        if err:
            return err
        if r.status_code >= 400:
            last = f'__ERROR__ {r.status_code} {r.text[:160]}'
            continue
        _API_FORMAT_CACHE[ck] = fmt
        try:
            return r.json()['choices'][0]['message']['content']
        except Exception as e:
            return f'__ERROR__ parse {e}'
    return last


def mock_rank(candidates, seed):
    """Simulate a discerning model: candidates WITH evidence float up, with noise."""
    rng = np.random.default_rng(seed)
    scored = [(1.0 * bool(c['evidence']) + rng.normal(0, 0.5), c['letter']) for c in candidates]
    scored.sort(key=lambda t: -t[0])
    ordered = [l for _, l in scored]
    items = ", ".join(
        f'{{"rank": {i+1}, "drug": "{l}", "basis": "both", '
        f'"evidence_agreement": "consistent", "confidence": 3, "rationale": "mock"}}'
        for i, l in enumerate(ordered))
    return f'Reasoning: mock.\n{{"ranking": [{items}]}}'


def rank_call(model, prompt, candidates, seed):
    """Single dispatch: mock / hosted-API / local-Ollama."""
    if MOCK:
        return mock_rank(candidates, SEED + seed)
    if is_api_model(model):
        if API_DELAY:
            time.sleep(API_DELAY)
        return api_rank(model, prompt, TEMPERATURE)
    # local model served by Ollama (bare tag, e.g. llama3.3:70b)
    return ollama_rank(model, prompt, SEED + seed, TEMPERATURE)


## 6 · ID crosswalks and resolvers

Gold-standard drugs (DrugBank) and diseases (MONDO) have to be mapped onto each
KG's own identifier scheme before we can look them up. These crosswalks bridge
DrugBank↔external drug IDs and MONDO↔DOID↔MESH disease IDs (the same logic the
KG loaders use), and are skipped entirely in mock mode.

In [ ]:
_BARE_MESH = re.compile(r'^[DC]\d+$')


def _id_variants(cands):
    out = set(cands)
    for c in list(cands):
        if not isinstance(c, str):
            continue
        if ':' in c:
            ns, _, rest = c.partition(':')
            out.update({c.replace(':', '_', 1), f'{ns.upper()}:{rest}', f'{ns.lower()}:{rest}', rest})
            try:
                stripped = str(int(rest))
                if stripped != rest:
                    out.add(stripped)
            except (ValueError, TypeError):
                pass
        if '_' in c and ':' not in c:
            out.add(c.replace('_', ':', 1))
    return out


def load_crosswalks():
    do        = pd.read_csv(GOLD / 'bridges' / 'do_diseases.csv')
    mesh_doid = pd.read_csv(GOLD / 'bridges' / 'mesh_to_doid.csv', on_bad_lines='skip')
    xref      = pd.read_csv(GOLD / 'bridges' / 'drugbank_xref.csv')
    sssom     = pd.read_csv(GOLD / 'bridges' / 'mondo.sssom.tsv', sep='\t', comment='#')
    sssom     = sssom[sssom['predicate_id'] == 'skos:exactMatch']

    _do = do.dropna(subset=['mondo_id', 'doid']).copy()
    _do['m'] = _do['mondo_id'].str.replace('MONDO:', '', regex=False).str.lstrip('0').replace({'': '0'})
    _do['d'] = _do['doid'].str.replace('DOID:', '', regex=False).str.lstrip('0').replace({'': '0'})
    mondo_to_doid_all, doid_to_mondo_all = defaultdict(set), defaultdict(set)
    for _, r in _do.iterrows():
        mondo_to_doid_all[r['m']].add(r['d'])
        doid_to_mondo_all[r['d']].add(r['m'])

    mesh_doid['mesh_clean'] = mesh_doid['mesh_id'].str.replace(r'\s*\{.*\}', '', regex=True).str.strip()
    _mh = (mesh_doid.assign(src_rank=mesh_doid['source'].map({'DO': 0, 'MONDO': 1}).fillna(2))
                    .sort_values('src_rank').drop_duplicates('mesh_clean', keep='first'))
    mesh_to_doid = {r['mesh_clean']: (r['doid'].replace('DOID:', '').lstrip('0') or '0')
                    for _, r in _mh.iterrows() if pd.notna(r['doid'])}
    doid_to_mesh = defaultdict(set)
    for m, d in mesh_to_doid.items():
        if m.startswith('MESH:'):
            doid_to_mesh[d].add(m)

    drug_equiv = defaultdict(set)
    for db, ns, eid in zip(xref.drugbank_id.astype(str), xref.namespace.astype(str), xref.external_id.astype(str)):
        drug_equiv[db].update({f'{ns}:{eid}', eid})
        if ns == 'PUBCHEM.COMPOUND':
            drug_equiv[db].update({f'CID:{eid}', f'pubchem.compound:{eid}'})

    _eq = defaultdict(set)
    for s, o in zip(sssom['subject_id'], sssom['object_id']):
        _eq[s].add(o); _eq[o].add(s)
    disease_equiv = defaultdict(set)
    for k, vs in _eq.items():
        s = set(vs)
        for v in vs:
            s |= _eq.get(v, set())
        s.discard(k)
        disease_equiv[k] = s

    return {'mondo_to_doid_all': mondo_to_doid_all, 'doid_to_mondo_all': doid_to_mondo_all,
            'mesh_to_doid': mesh_to_doid, 'doid_to_mesh': doid_to_mesh,
            'drug_equiv': drug_equiv, 'disease_equiv': disease_equiv}


def make_resolvers(config, kg_name, xw):
    drug_equiv        = xw['drug_equiv']
    mondo_to_doid_all = xw['mondo_to_doid_all']
    doid_to_mondo_all = xw['doid_to_mondo_all']
    mesh_to_doid      = xw['mesh_to_doid']
    doid_to_mesh      = xw['doid_to_mesh']
    disease_equiv     = xw['disease_equiv']

    def to_kg_drug_ids(db_id):
        s = str(db_id).strip()
        return _id_variants({s} | drug_equiv.get(s, set()))

    def to_kg_disease_ids(disease_id):
        s = str(disease_id).strip()
        scheme = config['knowledge_graphs'][kg_name].get('disease_id_scheme', 'doid')
        if   s.startswith('MONDO:'):
            mondo = s.replace('MONDO:', '').lstrip('0') or '0'
        elif s.startswith('DOID:'):
            mondo = sorted(doid_to_mondo_all.get(s.replace('DOID:', '').lstrip('0') or '0', ['?']))[0]
        elif _BARE_MESH.match(s):
            d = mesh_to_doid.get(f'MESH:{s}')
            mondo = sorted(doid_to_mondo_all.get(d, ['?']))[0] if d else '?'
        else:
            mondo = s.lstrip('0') or '0'
        cands = set()
        if scheme == 'mondo':
            cands.update({mondo, mondo.zfill(7), f'MONDO:{mondo}'})
        elif scheme == 'doid':
            for d in mondo_to_doid_all.get(mondo, set()):
                cands.update({d, f'DOID:{d}'})
        elif scheme == 'doid_mesh':
            for d in mondo_to_doid_all.get(mondo, set()):
                cands.update({d, f'DOID:{d}'}); cands |= doid_to_mesh.get(d, set())
        elif scheme == 'mesh':
            for d in mondo_to_doid_all.get(mondo, set()):
                for m in doid_to_mesh.get(d, set()):
                    cands.update({m, m.split(':', 1)[-1]})
        for c in list(cands) + [f'MONDO:{mondo}']:
            cands |= disease_equiv.get(c, set())
        return _id_variants(cands)

    return to_kg_drug_ids, to_kg_disease_ids


def build_edge_index(kg_df):
    ix = defaultdict(list)
    for x, y, r in zip(kg_df['x_index'].values, kg_df['y_index'].values, kg_df['relation'].values):
        u, v, rs = int(x), int(y), str(r)
        ix[u].append((v, rs)); ix[v].append((u, rs))
    return ix

## 7 · KG evidence block — entity-type dossiers (multi-KG)

Each candidate drug and the target disease are described by listing 1-hop edges as
`<subject> <phrase> <object>` triples. Edges are kept by the **entity type of the
neighbour** (resolved per-KG from `entity_types`): from a drug → gene neighbours are
`targets`; from a disease → gene neighbours are `associated gene`, plus related diseases
and pathways. The **drug↔disease bridge is excluded structurally** (a disease neighbour of
a drug is never rendered), so each KG's leak relation is handled automatically. `PK_EXCLUDE`
drops named ADME/pharmacokinetic relations per KG. Genes stay real (drug names are coded).

Gene IDs are normalised to HGNC symbols via an Entrez->symbol map (built from PrimeKG);
DRKG/OpenBioLink genes keyed by Entrez/NCBIGENE numbers become readable symbols. Any gene
that still has no symbol (non-coding / predicted loci absent from the map) is **dropped** —
a numeric gene ID carries no signal an LLM can use.


In [ ]:
# ── KG dossiers: entity-type policy (works across KGs) ──────────────────────
# Instead of hand-listing each KG's relation names (DRKG alone has ~26 drug→gene
# relations), edges are kept by the ENTITY TYPE of the neighbour, resolved per-KG
# from config entity_types. From a DRUG node: gene neighbours → "targets",
# pathway → "modulates pathway". From a DISEASE node: gene → "associated gene",
# pathway → "associated pathway", disease → "related disease". The drug↔disease
# bridge is excluded STRUCTURALLY (a disease neighbour of a drug, or vice-versa,
# is never rendered) so every KG's leak relation is handled automatically. The
# only per-KG list is PK_EXCLUDE: named ADME/pharmacokinetic relations whose gene
# neighbours are not mechanistic targets.
PK_EXCLUDE = {
    'primekg':    set(),
    'drkg':       {'DRUGBANK::enzyme::Compound:Gene', 'DRUGBANK::carrier::Compound:Gene',
                   'DRUGBANK::transporter::Compound:Gene'},
    'biokg':      {'DRUG_ENZYME', 'DRUG_TRANSPORTER', 'DRUG_CARRIER'},
    'openbilink': set(),
    'matrix':     set(),
    'hetionet':   set(),
}
DRUG_PHRASE    = {'gene': 'targets',              'pathway': 'modulates pathway'}
DISEASE_PHRASE = {'gene': 'is associated with gene', 'pathway': 'is associated with pathway',
                  'disease': 'is related to'}

def _gene_verb(role, rels):
    """Interaction/regulation verb for a drug/disease->gene edge, from the KG's
    relation name(s). Falls back to the generic phrase when the KG encodes no
    direction (e.g. PrimeKG 'drug_protein', BioKG 'DRUG_TARGET')."""
    s = ' '.join(str(r).lower() for r in rels)
    if role == 'drug':
        if any(k in s for k in ('inhibitor', 'antagonist', 'blocker', 'suppressor', '::n::', 'inhibits')): return 'inhibits'
        if any(k in s for k in ('agonist', 'activator', '::a+::', 'activates')):                            return 'activates'
        if any(k in s for k in ('cdg', 'downregulat', '::e-::')):                                           return 'downregulates'
        if any(k in s for k in ('cug', 'upregulat', '::e+::', 'inducer', 'induces')):                       return 'upregulates'
        if any(k in s for k in ('binder', 'binds', 'cbg', '::b::')):                                        return 'binds'
        if 'modulator' in s:                                                                               return 'modulates'
        return 'targets'
    else:
        if any(k in s for k in ('dug', 'upregulat', '::e+::')):   return 'upregulates'
        if any(k in s for k in ('ddg', 'downregulat', '::e-::')): return 'downregulates'
        return 'is associated with gene'


# Entrez gene-ID -> HGNC symbol map (built from PrimeKG, which keys genes by Entrez
# with the symbol as its name). Renders numeric gene IDs (DRKG 'Gene::3716',
# OpenBioLink 'NCBIGENE:3716') as readable symbols so the LLM can reason over them.
ENTREZ2SYM = {}
_ent_path = BASE / 'data' / 'gene_id_maps' / 'entrez_to_symbol.tsv'
if _ent_path.exists():
    for _line in open(_ent_path):
        _p = _line.rstrip('\n').split('\t')
        if len(_p) == 2:
            ENTREZ2SYM[_p[0]] = _p[1]


# A gene name that is only digits/separators is an unmapped Entrez ID (a non-coding
# or predicted locus absent from the symbol map). It is noise to the LLM, so such
# gene neighbours are dropped from dossiers (see _triples).
_UNREADABLE_GENE = re.compile(r'^[\d;,/|\s]+$')


def _entrez_syms(s):
    """Map a gene-node id to symbol(s); handles ';'-separated multi-gene nodes.
    'Gene::3716;3717' -> 'JAK1;JAK2'; returns None if nothing maps."""
    tail = str(s).split('::')[-1].split(':')[-1]
    syms = [ENTREZ2SYM[x] for x in re.split(r'[;,/|]', tail) if x.isdigit() and x in ENTREZ2SYM]
    return ';'.join(dict.fromkeys(syms)) if syms else None


# Heavy per-KG structures (graph, maps, resolvers) are cached so a cap/anonymize
# sweep reuses one load instead of re-reading the graph each time.
_KG_CACHE = {}


def _load_kg_core(kg_name):
    if kg_name in _KG_CACHE:
        return _KG_CACHE[kg_name]
    kg_df, nodes_df = load_kg(kg_name, config)
    mp = build_lookup_maps(nodes_df)
    name_map = mp['node_name_map']
    idx_to_type = dict(zip(nodes_df['idx'].astype(int), nodes_df['type'].astype(str)))
    et = config['knowledge_graphs'][kg_name]['entity_types']
    drug_t, disease_t = et.get('Drug'), et.get('Disease')
    gene_t = et.get('Gene/Protein')
    sep = config['knowledge_graphs'][kg_name].get('disease_id_separator')
    # kg-type string -> canonical role, for the entity-type policy
    canon = {}
    for role, kg_t in [('drug', et.get('Drug')), ('disease', et.get('Disease')),
                       ('gene', et.get('Gene/Protein')), ('pathway', et.get('Pathway'))]:
        if kg_t is not None:
            canon[str(kg_t)] = role

    id_to_idx = defaultdict(list)
    _ids   = nodes_df['id'].astype(str).tolist()
    _idxs  = nodes_df['idx'].astype(int).tolist()
    # Some KGs (OpenBioLink, MATRIX) key drug nodes by PubChem but carry a DrugBank
    # column; index those as alias keys so DrugBank-based gold IDs resolve.
    _db    = nodes_df['drugbank_id'].astype(str).tolist() if 'drugbank_id' in nodes_df.columns else None
    _dball = nodes_df['drugbank_ids_all'].astype(str).tolist() if 'drugbank_ids_all' in nodes_df.columns else None
    for i, (raw, idx) in enumerate(zip(_ids, _idxs)):
        nt = idx_to_type.get(int(idx))
        keys = {raw, raw.split('::', 1)[-1] if '::' in raw else raw}
        if sep and nt == disease_t and sep in raw:
            keys.update(p for p in raw.split(sep) if p)
        if _db is not None:
            v = _db[i]
            if v and v != 'nan':
                keys.add(v)
        if _dball is not None:
            v = _dball[i]
            if v and v != 'nan':
                keys.update(t.strip() for t in re.split(r'[|,;]', v) if t.strip())
        for k in keys:
            id_to_idx[k].append((int(idx), nt))

    # Normalise numeric gene IDs to HGNC symbols so the LLM can recognise them.
    if ENTREZ2SYM and gene_t is not None:
        _nfix = 0
        for raw, idx in zip(_ids, _idxs):
            if idx_to_type.get(int(idx)) == gene_t:
                sym = _entrez_syms(raw)
                if sym:
                    name_map[int(idx)] = sym
                    _nfix += 1
        if _nfix:
            print(f"  [{kg_name}] mapped {_nfix} numeric gene IDs -> symbols")

    to_drug_ids, to_disease_ids = make_resolvers(config, kg_name, xw=load_crosswalks())
    edge_index = build_edge_index(kg_df)
    degree = {u: len(adj) for u, adj in edge_index.items()}

    core = dict(name_map=name_map, idx_to_type=idx_to_type, gene_t=gene_t, canon=canon,
                drug_t=drug_t, disease_t=disease_t, id_to_idx=id_to_idx,
                to_drug_ids=to_drug_ids, to_disease_ids=to_disease_ids,
                edge_index=edge_index, degree=degree)
    _KG_CACHE[kg_name] = core
    return core


def make_kg_block_fn(kg_name, *, bridge_mode='full', cap=12, anonymize=False, anonymize_genes=False):
    """Return (drug_dossier_fn, disease_profile_fn) for one KG. Real mode only.
    Edges are kept by neighbour entity-type; drug↔disease edges are never rendered."""
    c = _load_kg_core(kg_name)
    name_map, idx_to_type, gene_t = c['name_map'], c['idx_to_type'], c['gene_t']
    drug_t, disease_t, canon = c['drug_t'], c['disease_t'], c['canon']
    id_to_idx = c['id_to_idx']
    to_drug_ids, to_disease_ids = c['to_drug_ids'], c['to_disease_ids']
    edge_index, degree = c['edge_index'], c['degree']
    pk = PK_EXCLUDE.get(kg_name, set())

    def _resolve(cands, want):
        for cand in cands:
            for idx, t in id_to_idx.get(cand, []):
                if t == want:
                    return idx
        return None

    _gene_code_maps = defaultdict(dict)

    def _gene_coder(disease_id):
        m = _gene_code_maps[disease_id]
        def code(name):
            if name not in m:
                m[name] = f"Gene-{len(m) + 1}"
            return m[name]
        return code

    def _triples(node_idx, subject, disease_id, role, exclude_idx=()):
        """One "<subject> <phrase> <object>" per kept neighbour. Gene edges render the
        KG's interaction/regulation modality where available (inhibits/activates/
        up-down-regulates/binds), else the generic phrase. Pathway and disease use
        fixed phrases. drug<->disease is excluded structurally (leakage)."""
        allowed = ('gene', 'pathway') if role == 'drug' else ('gene', 'pathway', 'disease')
        nb_rels, nb_role_of = defaultdict(set), {}
        for v, r in edge_index.get(node_idx, []):
            nb = canon.get(idx_to_type.get(v))
            if nb is None or nb not in allowed:
                continue
            if role == 'drug' and nb == 'gene' and str(r) in pk:   # drop ADME proteins
                continue
            nb_rels[v].add(r); nb_role_of[v] = nb
        groups = defaultdict(list)
        for v, rels in nb_rels.items():
            nb = nb_role_of[v]
            if nb == 'gene':
                phrase = _gene_verb(role, rels)
            elif nb == 'pathway':
                phrase = 'modulates pathway' if role == 'drug' else 'is associated with pathway'
            else:
                phrase = 'is related to'
            groups[phrase].append(v)
        coder = _gene_coder(disease_id) if anonymize_genes else None
        out = []
        for phrase in sorted(groups):
            seen, nbrs = set(exclude_idx), []
            for v in sorted(groups[phrase], key=lambda n: degree.get(n, 0)):
                if v in seen:
                    continue
                seen.add(v)
                if idx_to_type.get(v) == gene_t and _UNREADABLE_GENE.match(str(name_map.get(v, v))):
                    continue
                nbrs.append(v)
                if len(nbrs) >= cap:
                    break
            for v in nbrs:
                nm = name_map.get(v, str(v))
                if coder and idx_to_type.get(v) == gene_t:
                    nm = coder(nm)
                out.append(f"{subject} {phrase} {nm}")
        return "; ".join(out)

    def drug_dossier(drug_id, drug_name, disease_id):
        d_idx = _resolve(to_drug_ids(drug_id), drug_t)
        if d_idx is None:
            return None
        x_idx = _resolve(to_disease_ids(disease_id), disease_t)   # mask the target disease
        excl = {x_idx} if x_idx is not None else set()
        return _triples(d_idx, drug_name, disease_id, 'drug', excl) or None

    def disease_profile(disease_id, disease_name=None):
        if bridge_mode == 'mechanism_only':
            return ''
        x_idx = _resolve(to_disease_ids(disease_id), disease_t)
        if x_idx is None:
            return ''
        subject = disease_name or name_map.get(x_idx, 'disease')
        return _triples(x_idx, subject, disease_id, 'disease')

    return drug_dossier, disease_profile


## 8 · Candidate pools from the prospective gold standard

Each query is one disease whose true (post-cutoff) drug is hidden in a pool of
distractor drugs drawn from the rest of the gold set. Diseases are sampled
round-robin across therapeutic areas so the sample is not dominated by oncology,
and each drug is the positive for at most one query.

In [ ]:
def load_positives(gold_file):
    df = pd.read_csv(gold_file, sep='\t')
    if 'drug_id' not in df.columns and 'drugbank_id' in df.columns:
        df = df.rename(columns={'drugbank_id': 'drug_id'})
    if 'disease_id' not in df.columns and 'mondo_id' in df.columns:
        df = df.rename(columns={'mondo_id': 'disease_id'})
    need = {'drug_id', 'drug_name', 'disease_name', 'disease_id'}
    missing = need - set(df.columns)
    if missing:
        raise SystemExit(f"{gold_file.name} missing columns: {missing}")
    cols = ['drug_id', 'drug_name', 'disease_name', 'disease_id']
    out = df[cols + (['therapeutic_area'] if 'therapeutic_area' in df.columns else [])].dropna(subset=cols)
    if 'therapeutic_area' not in out.columns:
        out = out.assign(therapeutic_area='unknown')
    return out.reset_index(drop=True)


# ── Fixed distractor pool: Open Targets approved-drug universe ───────────────
_XW_POOL = load_crosswalks()
def _disease_to_mondos(disease_id):
    s = str(disease_id).strip()
    if s.startswith('MONDO:'):
        return {s.replace('MONDO:', '').lstrip('0') or '0'}
    if s.startswith('DOID:'):
        return set(_XW_POOL['doid_to_mondo_all'].get(s.replace('DOID:', '').lstrip('0') or '0', []))
    d = re.sub(r'\D', '', s).lstrip('0')
    return {d} if d else set()

def load_ot_distractor_pool():
    ot = pd.read_csv(GOLD / 'bridges' / 'opentargets_drug_indications.csv')
    appr = ot[ot['is_approved'] == True].dropna(subset=['drugbank_id'])
    universe = (appr.drop_duplicates('drugbank_id')[['drugbank_id', 'drug_name']]
                    .rename(columns={'drugbank_id': 'drug_id'}).reset_index(drop=True))
    indicated = defaultdict(set)
    for col in ['mondo_id', 'mondo_id_mapped']:
        if col in ot.columns:
            for db, m in zip(ot['drugbank_id'].astype(str), ot[col]):
                if pd.notna(m):
                    try:
                        indicated[str(int(float(m)))].add(db)
                    except (ValueError, TypeError):
                        pass
    return universe, indicated

_OT_UNIVERSE, _OT_INDICATED = load_ot_distractor_pool()
print(f'distractor pool: Open Targets approved drugs = {len(_OT_UNIVERSE)} ; diseases with indication lists = {len(_OT_INDICATED)}')


def build_queries(pos_df, n_diseases, pool_size, seed, *, unique_drug=True, stratify=True,
                  per_pair=False, distractor_pool='opentargets_approved'):
    rng = np.random.default_rng(seed)
    df = pos_df.copy()
    gold_drugs = df.drop_duplicates('drug_id')[['drug_id', 'drug_name']]
    pos_by_disease = df.groupby('disease_id')['drug_id'].agg(set).to_dict()
    if per_pair:
        sel = df.reset_index(drop=True)
    else:
        if unique_drug:
            df = df.drop_duplicates('drug_id')
        sel = df.drop_duplicates('disease_name').reset_index(drop=True)
    n = min(n_diseases, len(sel))
    if stratify and sel['therapeutic_area'].nunique() > 1:
        areas = sorted(sel['therapeutic_area'].unique())
        pools = {a: list(rng.permutation(sel.index[sel['therapeutic_area'] == a].to_numpy())) for a in areas}
        picks = []
        while len(picks) < n and any(pools.values()):
            for a in areas:
                if pools[a]:
                    picks.append(int(pools[a].pop()))
                    if len(picks) >= n:
                        break
        chosen = sel.loc[picks]
    else:
        chosen = sel.iloc[rng.choice(len(sel), size=n, replace=False)]
    use_ot = (distractor_pool == 'opentargets_approved')
    queries = []
    for _, row in chosen.iterrows():
        excl = set(pos_by_disease.get(row['disease_id'], set())) | {row['drug_id']}
        for mm in _disease_to_mondos(row['disease_id']):
            excl |= _OT_INDICATED.get(mm, set())
        pool = (_OT_UNIVERSE if use_ot else gold_drugs)
        pool = pool[~pool['drug_id'].isin(excl)]
        k = min(pool_size - 1, len(pool))
        dis = pool.iloc[rng.choice(len(pool), size=k, replace=False)]
        cands = [{'drug_id': row['drug_id'], 'drug_name': row['drug_name'], 'is_pos': True}]
        cands += [{'drug_id': d.drug_id, 'drug_name': d.drug_name, 'is_pos': False} for d in dis.itertuples()]
        queries.append({'disease_name': row['disease_name'], 'disease_id': row['disease_id'],
                        'area': row['therapeutic_area'], 'candidates': cands})
    return queries


def assign_letters_and_evidence(cands, block_for, disease_id, condition, mock, rng, code_map=None):
    """Attach a letter and (for KG conditions) an evidence string to each candidate.
    Order is shuffled here. Returns (candidate_view, positive_letter)."""
    order = list(range(len(cands)))
    rng.shuffle(order)
    view, pos_letter = [], None
    for new_i, orig_i in enumerate(order):
        c = cands[orig_i]
        letter = LETTERS[new_i]
        name = code_map[c['drug_id']] if code_map else c['drug_name']
        evidence = None
        if condition != 'no_kg':
            if mock:
                p = 0.9 if c['is_pos'] else 0.4
                evidence = "mechanistic link present" if rng.random() < p else None
            else:
                evidence = block_for(c['drug_id'], name, disease_id)
        if c['is_pos']:
            pos_letter = letter
        view.append({'letter': letter, 'drug_name': name, 'evidence': evidence})
    return view, pos_letter


pos_df = load_positives(GOLD_FILE)
QUERIES = build_queries(pos_df, N_DISEASES, POOL_SIZE, SEED,
                        unique_drug=UNIQUE_DRUG, stratify=STRATIFY, per_pair=PER_PAIR,
                        distractor_pool=DISTRACTOR_POOL)
print(f'{len(QUERIES)} queries  (pool={POOL_SIZE}):')
for q in QUERIES:
    print(f"  - [{q['area']}] {q['disease_name']} ({q['disease_id']})  "
          f"true drug: {q['candidates'][0]['drug_name']}")


## 9 · Ranking loop (idempotent)

For every query × condition × model × shuffle we build the prompt, call the model,
parse the ranking, and record the rank of the true drug plus the reliance fields.
The no-KG baseline runs once; each KG arm loads its graph one at a time. Results
are cached to `results/tables/09_llm_runs/09_ranking.csv` — rerun with
`FORCE_RERUN = True` to recompute.

In [ ]:
def _code_map(q):
    if not ANONYMIZE:
        return None
    ids = sorted({c['drug_id'] for c in q['candidates']})
    return {did: f"Drug-{i + 1}" for i, did in enumerate(ids)}


def run_arm(kg_tag, block_for, disease_profile_fn, conds, queries=None, base_seed=None):
    """Model loop for `conds`; rows tagged with kg_tag (or 'none' for no_kg).
    `queries`/`base_seed` default to the module-level QUERIES/SEED."""
    queries = QUERIES if queries is None else queries
    base = SEED if base_seed is None else base_seed
    out, n_pos_ev, n_cells = [], 0, 0
    rl_streak = 0
    total = len(queries) * len(conds) * len(MODELS) * SHUFFLES
    pbar = tqdm(total=total, desc=f'seed={base} arm={kg_tag}', leave=False)
    _WORKED_EXAMPLES = {}   # (kg, condition) -> first prompt, dumped verbatim
    for qi, q in enumerate(queries):
        code_map = _code_map(q)
        for cond in conds:
            for model in MODELS:
                for s in range(SHUFFLES):
                    rng = np.random.default_rng(base + 1000 * qi + 100 * s
                                                + hash((cond, model, kg_tag)) % 97)
                    view, pos_letter = assign_letters_and_evidence(
                        q['candidates'], block_for, q['disease_id'], cond, MOCK, rng, code_map=code_map)
                    if cond != 'no_kg':
                        n_cells += 1
                        pos_ev = next(c['evidence'] for c in view if c['letter'] == pos_letter)
                        n_pos_ev += int(bool(pos_ev))
                    dprof = (disease_profile_fn(q['disease_id'], q['disease_name'])
                             if (cond != 'no_kg' and disease_profile_fn) else None)
                    prompt = build_prompt(q['disease_name'], q['disease_id'], view, disease_profile=dprof)
                    # First prompt per (KG, condition) is saved verbatim for the
                    # worked example in the Supplementary Information.
                    _ex_key = f'{kg_tag}__{cond}'
                    if _ex_key not in _WORKED_EXAMPLES:
                        _WORKED_EXAMPLES[_ex_key] = prompt
                        (TABLES / f'09_worked_example_{_ex_key}.txt').write_text(
                            f'# {q["disease_name"]} ({q["disease_id"]})  kg={kg_tag}  '
                            f'condition={cond}  positive={pos_letter}\n\n' + prompt + '\n')
                    valid = [c['letter'] for c in view]
                    resp = rank_call(model, prompt, view, s + qi)
                    err = (resp[9:].strip()[:120] if isinstance(resp, str)
                           and resp.startswith('__ERROR__') else None)
                    rl_streak = rl_streak + 1 if (err and '429' in err) else 0
                    ordered, fields = parse_response(resp, valid)
                    r = rank_of(pos_letter, ordered, POOL_SIZE)
                    bvals = [v['basis'] for v in fields.values() if v.get('basis')]
                    frac_kg = (sum(b in ('kg', 'both') for b in bvals) / len(bvals) if bvals else None)
                    confs = [v['confidence'] for v in fields.values() if isinstance(v.get('confidence'), int)]
                    pf = fields.get(pos_letter, {})
                    out.append({
                        'seed': base, 'disease': q['disease_name'], 'kg': (kg_tag if cond != 'no_kg' else 'none'),
                        'condition': cond, 'model': model, 'shuffle': s,
                        'pool_size': POOL_SIZE, 'positive_letter': pos_letter,
                        'rank': r, 'reciprocal_rank': 1.0 / r,
                        'hit@1': int(r <= 1), 'hit@3': int(r <= 3), 'hit@5': int(r <= 5),
                        'parsed': int(bool(ordered)), 'error': err,
                        'pos_basis': pf.get('basis'), 'pos_agreement': pf.get('evidence_agreement'),
                        'pos_confidence': pf.get('confidence'), 'pos_rationale': pf.get('rationale'),
                        'frac_basis_kg': frac_kg,
                        'mean_confidence': (sum(confs) / len(confs) if confs else None),
                    })
                    pbar.update(1)
                    if rl_streak >= 8:
                        pbar.close()
                        print("\nRate limit exhausted (8 calls failed in a row); aborting early "
                              "with partial results. Retry later or switch MODELS to another provider.")
                        return out, n_pos_ev, n_cells
    pbar.close()
    return out, n_pos_ev, n_cells


import glob as _glob, os as _os
_compiled = sorted(_glob.glob(str(OUT_CSV.parent / '09_big_*.csv')))
if LOAD_COMPILED and _compiled:
    res = pd.concat([pd.read_csv(f) for f in _compiled], ignore_index=True)
    print(f'LOAD_COMPILED: {len(res)} rows from {len(_compiled)} per-LLM files '
          f'({", ".join(_os.path.basename(f) for f in _compiled)})')
elif OUT_CSV.exists() and not FORCE_RERUN:
    res = pd.read_csv(OUT_CSV)
    print(f'loaded cached results ({len(res)} rows) from {OUT_CSV.name}')
else:
    if not MOCK:
        ollama_wanted = [m for m in MODELS if not is_api_model(m)]
        if ollama_wanted:
            avail = ollama_models_available()
            missing = [m for m in ollama_wanted if m not in avail]
            if missing:
                print(f'  not pulled, skipping: {missing}')
                MODELS = [m for m in MODELS if is_api_model(m) or m in avail]
        for m in [m for m in MODELS if is_api_model(m)]:
            if not _api_key_for(m):
                print(f'  no API key for {m}; skipping')
                MODELS = [x for x in MODELS if x != m]
        if not MODELS:
            raise SystemExit('No usable models: pull an Ollama model or set an API key.')

    rows, coverage = [], {}
    # KG block fns are seed-independent -> build once per KG and reuse across seeds.
    kg_fns = {}
    if not MOCK and any(c != 'no_kg' for c in CONDITIONS):
        for kg in KGS:
            print(f"Loading KG '{kg}' (cap={CAP}, anonymize={ANONYMIZE}) ...")
            kg_fns[kg] = make_kg_block_fn(kg, bridge_mode=BRIDGE_MODE, cap=CAP,
                                          anonymize=ANONYMIZE, anonymize_genes=ANONYMIZE_GENES)
    for seed in SEEDS:
        Q = build_queries(pos_df, N_DISEASES, POOL_SIZE, seed,
                          unique_drug=UNIQUE_DRUG, stratify=STRATIFY, per_pair=PER_PAIR, distractor_pool=DISTRACTOR_POOL)
        print(f"\n=== seed {seed}: {len(Q)} diseases ===")
        if MOCK:
            rr, npev, ncell = run_arm('mock', None, None, CONDITIONS, queries=Q, base_seed=seed)
            rows += rr; continue
        if 'no_kg' in CONDITIONS:
            rr, _, _ = run_arm('none', None, None, ['no_kg'], queries=Q, base_seed=seed); rows += rr
        for kg in KGS:
            bf, dpf = kg_fns[kg]
            rr, npev, ncell = run_arm(kg, bf, dpf, ['kg'], queries=Q, base_seed=seed)
            rows += rr; coverage[(kg, seed)] = (npev, ncell)

    res = pd.DataFrame(rows)
    res.to_csv(OUT_CSV, index=False)
    print(f'\nwrote {len(res)} rows -> {OUT_CSV}')
    if any(nc for _, nc in coverage.values()):
        print('positive coverage by KG (cells where the true drug had a dossier):')
        for key, (npev, ncell) in coverage.items():
            if ncell:
                print(f'  {str(key):<22} {npev}/{ncell}')


## 10 · Results — MRR / hits@k per model × KG × condition

Higher MRR and hits@k mean the true drug is ranked nearer the top. `reliance` is
the mean fraction of candidates the model said it judged using KG evidence
(`basis` ∈ {kg, both}); `conf` is mean self-reported confidence; `parse_rate` is
the fraction of responses that yielded a valid ranking.

In [ ]:
summary = (res.groupby(['model', 'kg', 'condition'])
           .agg(MRR=('reciprocal_rank', 'mean'),
                hit1=('hit@1', 'mean'), hit3=('hit@3', 'mean'), hit5=('hit@5', 'mean'),
                reliance=('frac_basis_kg', 'mean'), conf=('mean_confidence', 'mean'),
                parse_rate=('parsed', 'mean'), n=('rank', 'size'))
           .round(3).reset_index().sort_values(['model', 'condition', 'kg']))
if 'error' in res and res['error'].notna().any():
    n_err = int(res['error'].notna().sum())
    print(f'note: {n_err}/{len(res)} calls errored (counted as misses, they deflate MRR)')
summary

---
## Headline figures
Generated from the run data above via `src/headline_figures.py` (see also `09b_headline_figures.ipynb`).


In [ ]:
import os, sys
from IPython.display import display
%matplotlib inline
BASE = os.getcwd()
while not os.path.exists(os.path.join(BASE, 'src', 'headline_figures.py')) and os.path.dirname(BASE) != BASE:
    BASE = os.path.dirname(BASE)
sys.path.insert(0, BASE)
from src.headline_figures import (load_runs, fig_kg_lift_bars, fig_lift_box,
    fig_rank_distribution, fig_dim_tracks_lift, fig_lift_vs_coverage, fig_evidence_quality)
_RUNS = os.path.join(BASE, 'results', 'tables', '09_llm_runs')
_SUM  = os.path.join(BASE, 'results', 'tables', '00_benchmark_summary.csv')
_hf_df = load_runs(_RUNS)
print(f'loaded {len(_hf_df)} parsed rows | models: {sorted(_hf_df.Model.unique())}')


In [ ]:
fig_kg_lift_bars(_hf_df);


In [ ]:
fig_lift_box(_hf_df);


In [ ]:
fig_rank_distribution(_hf_df);


In [ ]:
fig_dim_tracks_lift(_hf_df, _SUM);


In [ ]:
fig_lift_vs_coverage(_hf_df, _SUM);


In [ ]:
fig_evidence_quality(_hf_df);
